This application will convert text from english to other language - it's just simple LLM call plus prompting, but still great - a lot of things can be made using this
LCEL - Langchain expression language

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [16]:
from langchain_groq import ChatGroq
model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x12b426920>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12c7def80>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from langchain_core.messages import HumanMessage, SystemMessage
# we need to specify to llm that which message is provided by user
# and which message is a kind of instruction to the llm model

messages =[
    SystemMessage(content= "Translate the following from English to French"),
    HumanMessage(content= "Hello how are you?")

]
model.invoke(messages)

AIMessage(content='Bonjour, comment allez-vous?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 27, 'total_tokens': 35, 'completion_time': 0.018513693, 'completion_tokens_details': None, 'prompt_time': 0.000152328, 'prompt_tokens_details': None, 'queue_time': 0.045037058, 'total_time': 0.018666021}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d488d-e4a0-7132-946e-b3a5528df57e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 8, 'total_tokens': 35})

In [20]:
result = model.invoke(messages)

In [21]:
# use string output parser because I dont want it as AI Response
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous?'

In [22]:
# Using LCEL we can chain the components

chain = model | parser
chain.invoke(messages)
# it is going to first give messages(both) to model and will get the AI response
# then this output will go to parser and I will get the output

'Bonjour, comment allez-vous?'

In [23]:
# intead of passing the list of messages we can do 1 efficient message
# Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    ["system",generic_template, ("user","{text}")]
)

In [26]:
result =prompt.invoke({"language": "French", "text":"Hello"})
# it is doing the same thing converting prompt into system message and human message

In [27]:
result.to_messages()

[HumanMessage(content='system', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [31]:
chain = prompt|model|parser
chain.invoke({"language":"French","text":"It is a good day"})
# giving the 2 parameters to it

'The translation of "It is a good day" into French is:\n\n"C\'est une bonne journée"'

In [ ]:
# We can deploy entire langchain runnables and chains as REST API using Langserve
